# Structure-Validated De Novo Binder Design

End-to-end pipeline: **BoltzGen → LigandMPNN → BoltzFold**

1. **BoltzGen** generates de novo backbones conditioned on a target
2. **LigandMPNN** designs sequences for each backbone
3. **BoltzFold** refolds each designed sequence to validate structure and binding

**Requirements:**
- CUDA-12 GPU (≥16 GB VRAM recommended; 32 GB system RAM minimum)
- Python 3.12 environment with `evedesign[boltzgen,boltz2fold-cuda,mpnn]` installed
- See `docs/install-gpu.md` for the install workflow

In [ ]:
import torch
from loguru import logger

from evedesign.system import System, Protein
from evedesign.models.boltzgen import BoltzGenGenerator
from evedesign.models.mpnn import LigandMPNN
from evedesign.models.boltzfold import BoltzFoldTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"BoltzGen available: {BoltzGenGenerator.available}")
print(f"LigandMPNN available: {LigandMPNN.available}")
print(f"BoltzFold available: {BoltzFoldTransformer.available}")

if DEVICE == "cpu":
    raise RuntimeError(
        "CUDA GPU required. See docs/install-gpu.md."
    )

## Step 1 — Define target and binder

**De novo binder design** means: design a brand new protein that binds to a known target. BoltzGen is a *conditional* generator — it doesn't make proteins in a vacuum, it makes them conditioned on a target.

This notebook uses **1G13** (human GM2 activator protein, chain A) as the target — the canonical test case from BoltzGen's `vanilla_protein/1g13prot.yaml` example. The target CIF is downloaded from RCSB on first run and cached locally.

System layout:

- **Target** (`rep=<1G13 chain A sequence>`, `structures={"input": <Structure>}`): the protein BoltzGen designs against. Both sequence and structure are attached.
- **Binder** (`rep=None`, `min_length=80`, `max_length=140`): the de novo binder. No starting sequence, no starting backbone — BoltzGen invents both. The length range matches BoltzGen's vanilla binder convention.

To swap in a different target, change `TARGET_PDB_ID` and `TARGET_CHAIN` below.

In [ ]:
import urllib.request
from pathlib import Path

import biotite.structure as struc

from evedesign.structure import Structure, StructureFile

# ─── Target configuration ───────────────────────────
TARGET_PDB_ID = "1G13"          # GM2 activator protein
TARGET_CHAIN = "A"
BINDER_MIN_LENGTH = 80           # BoltzGen vanilla default
BINDER_MAX_LENGTH = 140

# Pipeline sizing
NUM_BACKBONES = 2                # backbones from BoltzGen
NUM_SEQUENCES_PER_BACKBONE = 4   # MPNN sequences per backbone

# ─── Download target CIF (cached locally) ───────────
CACHE_DIR = Path("./target_cache")
CACHE_DIR.mkdir(exist_ok=True)
cif_path = CACHE_DIR / f"{TARGET_PDB_ID.lower()}.cif"

if not cif_path.exists():
    url = f"https://files.rcsb.org/download/{TARGET_PDB_ID}.cif"
    print(f"Downloading {TARGET_PDB_ID} from RCSB...")
    urllib.request.urlretrieve(url, cif_path)
    print(f"  → cached at {cif_path}")
else:
    print(f"Using cached {TARGET_PDB_ID} from {cif_path}")

# ─── Load and filter target chain ───────────────────
sf = StructureFile(str(cif_path), format="cif")
model = sf.get_model()
chain = model.get_chain(TARGET_CHAIN)

# Filter to amino acids only — drops ligands, waters,
# ions that share the chain ID
aa_mask = struc.filter_amino_acids(chain.atom_array)
target_structure = Structure(chain.atom_array[aa_mask])

# Sanity check: no non-AA residues remain
res_df = target_structure.res_df()
assert not res_df.res_name_oneletter.isnull().any(), (
    f"Filtered chain {TARGET_CHAIN} still has non-AA "
    "residues — investigate the CIF"
)

target_sequence = "".join(res_df.res_name_oneletter)
print(f"Target: {TARGET_PDB_ID} chain {TARGET_CHAIN}, "
      f"{len(target_sequence)} aa")
print(f"  seq: {target_sequence[:60]}"
      f"{'...' if len(target_sequence) > 60 else ''}")

# ─── Build System ───────────────────────────────────
system = System([
    Protein(
        rep=target_sequence,
        id="target",
        structures={"input": target_structure},
    ),
    Protein(
        rep=None,
        min_length=BINDER_MIN_LENGTH,
        max_length=BINDER_MAX_LENGTH,
        id="binder",
    ),
])

print(f"\nSystem: {len(system)} entities")
for i, entity in enumerate(system):
    rep_summary = (
        f"fixed ({len(entity.rep)} aa)"
        if entity.rep is not None
        else f"design ({entity.min_length}..{entity.max_length})"
    )
    has_struct = (
        "with structure"
        if entity.structures
        else "no structure"
    )
    print(f"  Entity {i} ({entity.id}): {rep_summary}, "
          f"{has_struct}")

## Step 2: Generate backbones with BoltzGen

Run BoltzGen's diffusion model to produce candidate binder backbones. The pipeline:
1. Writes a BoltzGen YAML spec for the system
2. Invokes the `boltzgen run` CLI
3. Parses outputs into `SystemInstance` objects with structures on `EntityInstance.models["model_0"]`

In [ ]:
generator = BoltzGenGenerator(
    protocol="protein-anything",
    device=DEVICE,
    skip_inverse_folding=True,
    budget=2,
    keep_tmp_dir=True,
).build(system)

print(f"Generator ready: {generator.ready}")
print(f"Generating {NUM_BACKBONES} backbones (5-20 min on first run for checkpoint downloads)...")

backbones = generator.generate(num_designs=NUM_BACKBONES)

print(f"\nGenerated {len(backbones)} backbones:")
for i, bb in enumerate(backbones):
    metrics = bb.metadata.get("boltzgen_metrics", {})
    print(f"  Backbone {i}: id={bb.metadata.get('boltzgen_design_id')}, "
          f"score={bb.score:.3f}, "
          f"confidence={bb.confidence:.3f}")
    print(f"    binder seq: {''.join(bb[1].rep[:20])}... "
          f"(len={len(bb[1].rep)})")
    print(f"    iptm={metrics.get('iptm', 'n/a'):.3f}, "
          f"ptm={metrics.get('ptm', 'n/a'):.3f}, "
          f"complex_plddt={metrics.get('complex_plddt', 'n/a'):.3f}")

### Visualize BoltzGen backbones

Each backbone is a target + designed-binder complex. The target is on the left chain, the binder on the right. Coloring is by per-residue pLDDT — red (50) is low confidence, blue (90+) is high.

In [ ]:
!pip install py3Dmol -q

import io
import py3Dmol

def view_complex(instance, title=None, width=800, height=400):
    """
    Render all entities of a SystemInstance as a single
    py3Dmol view, one addModel per entity. Coloring by
    per-residue pLDDT (B-factor) with the roygb gradient.
    """
    view = py3Dmol.view(width=width, height=height)

    for entity_idx in range(len(instance)):
        ei = instance[entity_idx]
        if ei.models is None or "model_0" not in ei.models:
            continue
        structure = ei.models["model_0"]
        if isinstance(structure, list):
            structure = structure[0]  # homo-oligomer: take first

        buf = io.StringIO()
        structure.to_file(buf, format="cif")
        view.addModel(buf.getvalue(), "cif")

    view.setStyle({
        "cartoon": {
            "colorscheme": {
                "prop": "b",
                "gradient": "roygb",
                "min": 50,
                "max": 90,
            }
        }
    })
    view.zoomTo()
    if title:
        print(title)
    view.show()
    return view

for i, bb in enumerate(backbones):
    metrics = bb.metadata.get("boltzgen_metrics", {})
    title = (
        f"BoltzGen backbone {i} "
        f"(id={bb.metadata.get('boltzgen_design_id')}, "
        f"iptm={metrics.get('iptm', 0):.3f}, "
        f"plddt={metrics.get('complex_plddt', 0):.3f})"
    )
    view_complex(bb, title=title)

## Step 3: Design sequences with LigandMPNN

For each BoltzGen backbone:
1. Adapt the backbone's structure into a template-level System via `System.with_instance_structures`
2. Build a LigandMPNN model on that backbone-system
3. Generate `NUM_SEQUENCES_PER_BACKBONE` sequence designs per backbone

Each backbone produces its own batch of designed sequences — collect them all into a single list for the refolding step.

In [ ]:
designed_instances = []

for bb_idx, backbone in enumerate(backbones):
    print(f"--- Backbone {bb_idx} → MPNN ---")

    # Adapter: promote backbone.models["model_0"] onto Entity.structures for MPNN to consume
    bb_system = system.with_instance_structures(backbone)

    mpnn = LigandMPNN(
        model_name="ligandmpnn_v_32_010_25",
        device=DEVICE,
    ).build(bb_system)

    seqs = mpnn.generate(
        num_designs=NUM_SEQUENCES_PER_BACKBONE,
        temperature=0.1,
    )

    # Tag each design with its parent backbone for downstream tracing
    for seq_idx, design in enumerate(seqs):
        design.metadata["parent_backbone_idx"] = bb_idx
        design.metadata["parent_backbone_id"] = (
            backbone.metadata.get("boltzgen_design_id")
        )
        design.metadata["mpnn_seq_idx"] = seq_idx

    designed_instances.extend(seqs)
    print(f"  → {len(seqs)} sequences designed")
    for d in seqs:
        print(f"    score={d.score:.3f}: {''.join(d[1].rep[:30])}...")

print(f"Total designed instances: {len(designed_instances)}")

## Step 4: Refold and validate with BoltzFold

Each MPNN-designed sequence is refolded by BoltzFold to confirm:
1. The sequence folds into a structure resembling the intended backbone (low RMSD to the BoltzGen design)
2. The complex still has high interface confidence (ipTM)

BoltzFold returns one SystemInstance per input, with `diffusion_samples` ranks stored as `models["model_0".."model_N-1"]`. The best rank is `model_0`; `instance.score` and `instance.confidence` are scalars from that best rank.

In [ ]:
folder = BoltzFoldTransformer(
    device=DEVICE,
    sampling_steps=200,
    diffusion_samples=3,   # set higher in production (5 is BoltzFold's default)
    recycling_steps=3,
).build(system)

print(f"Refolding {len(designed_instances)} designs (this may take several minutes)...")

refolded = folder.transform(designed_instances)

print(f"Refolded {len(refolded)} designs:")
for i, r in enumerate(refolded):
    parent = r.metadata.get("parent_backbone_idx", "?")
    print(f"  Design {i} (from backbone {parent}): "
          f"score={r.score:.3f}, "
          f"confidence={r.confidence:.3f}")

### Visualize BoltzFold refolds

Each refold shows whether the MPNN-designed sequence folds back into something resembling its BoltzGen parent backbone. The complex is colored by pLDDT — a clean blue cartoon means the refold is confident in the predicted structure.

In [ ]:
# Reuses view_complex defined above
for i, r in enumerate(refolded):
    parent = r.metadata.get("parent_backbone_idx", "?")
    title = (
        f"Refold {i} (from backbone {parent}): "
        f"score={r.score:.3f}, "
        f"confidence={r.confidence:.3f}"
    )
    view_complex(r, title=title)

## Step 5: Filter and rank

Filter the refolded designs by structural confidence:
- `confidence > 0.7` (complex pLDDT — overall structure confidence)
- `score > 0.5` (the configured score_attribute, by default `confidence_score`)

Then sort by score descending. In production you'd also filter by interface metrics (ipTM, ipSAE) and liability counts.

In [ ]:
import pandas as pd

PLDDT_THRESHOLD = 0.7
SCORE_THRESHOLD = 0.5

rows = []
for i, r in enumerate(refolded):
    rows.append({
        "design_idx": i,
        "parent_backbone": r.metadata.get("parent_backbone_idx"),
        "parent_backbone_id": r.metadata.get("parent_backbone_id"),
        "score": r.score,
        "confidence": r.confidence,
        "sequence": "".join(r[1].rep) if r[1].rep is not None else "",
        "length": len(r[1].rep) if r[1].rep is not None else 0,
    })

df = pd.DataFrame(rows)
print("All designs:")
print(df.to_string(index=False))

passing = df[
    (df["confidence"] > PLDDT_THRESHOLD)
    & (df["score"] > SCORE_THRESHOLD)
].sort_values("score", ascending=False)

print(f"{len(passing)}/{len(df)} designs passed "
      f"(confidence > {PLDDT_THRESHOLD}, "
      f"score > {SCORE_THRESHOLD}):")
if len(passing) > 0:
    print(passing.to_string(index=False))
else:
    print("None, try lowering thresholds, increasing NUM_BACKBONES, or improving the target.")

## Side-by-side: BoltzGen backbone vs BoltzFold refold

For the top-scoring refold, visualize the BoltzGen backbone (left) next to its BoltzFold refold (right). A successful design has matching topology in both — the refold should look like a slightly cleaner version of the same structure.

In [ ]:
# Pick the top-scoring refold
if len(refolded) == 0:
    print("No refolds to compare")
else:
    top_idx = max(
        range(len(refolded)),
        key=lambda i: refolded[i].score or 0.0,
    )
    top_refold = refolded[top_idx]
    parent_idx = top_refold.metadata.get("parent_backbone_idx", 0)
    parent_backbone = backbones[parent_idx]

    print(f"Top refold: design {top_idx} "
          f"(from backbone {parent_idx})")
    print(f"  refold score:      {top_refold.score:.3f}")
    print(f"  refold confidence: {top_refold.confidence:.3f}")
    print(f"  backbone score:    {parent_backbone.score:.3f}")
    print(f"  backbone conf:     {parent_backbone.confidence:.3f}")

    print("\n[Left] BoltzGen backbone:")
    view_complex(parent_backbone)

    print("\n[Right] BoltzFold refold:")
    view_complex(top_refold)